In [1]:
# from networks import ConvNet
import numpy as np
import torch
from torch.autograd import Variable
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets
import torch.nn.functional as F
import torchvision.transforms as transforms
from tqdm import tqdm
import time
import copy

device=0

torch.manual_seed(1222)

In [2]:
transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))])

train_set = torchvision.datasets.MNIST(root='../data', train=True, download=True, transform=transform)
# batch_size = len(train_set) // args.nworker
train_loader = DataLoader(train_set)
test_loader = DataLoader(torchvision.datasets.MNIST(root='../data', train=False, download=True, transform=transform))

# network = ConvNet(input_size=28, input_channel=1, classes=10, filters1=30, filters2=30, fc_size=200).to(device)

In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv_1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=5, stride=1)
        self.conv_2 = nn.Conv2d(in_channels=4, out_channels=10, kernel_size=5, stride=1)
        self.fc_1 = nn.Linear(in_features=4 * 4 * 10, out_features=100)
        self.fc_2 = nn.Linear(in_features=100, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv_1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv_2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 4 * 4 * 10)
        x = F.relu(self.fc_1(x))
        x = self.fc_2(x)
        return x

In [4]:
network = Net().to(device)

In [5]:
# poisoning rate (defined in paper)
alpha = 2
# alpha=0

# do clustering to get a subpop
k=100 # recommended param in paper

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=k)

x_train = torch.stack([i[0][0] for i in train_loader]).to(device)
y_train = torch.stack([i[1][0] for i in train_loader]).type(torch.long).to(device)



In [6]:
# params (LR) recommended in paper
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(network.parameters(), lr=0.005)

In [7]:
# train a base model with 20% of the training data, then we fine tune on the poisoned data
# this mirrors the threat model assumed by the paper

n_epoch = 100
batch_size = 100
train_set_size = len(x_train)//2
for _ in range(n_epoch):
    print("epoch " + str(_))
    
    for i in range(0,train_set_size,batch_size):

    
        feature = x_train[i:i+batch_size]
        target = y_train[i:i+batch_size]
        optimizer.zero_grad()
        output = network(feature)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

epoch 0
epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10
epoch 11
epoch 12
epoch 13
epoch 14
epoch 15
epoch 16
epoch 17
epoch 18
epoch 19
epoch 20
epoch 21
epoch 22
epoch 23
epoch 24
epoch 25
epoch 26
epoch 27
epoch 28
epoch 29
epoch 30
epoch 31
epoch 32
epoch 33
epoch 34
epoch 35
epoch 36
epoch 37
epoch 38
epoch 39
epoch 40
epoch 41
epoch 42
epoch 43
epoch 44
epoch 45
epoch 46
epoch 47
epoch 48
epoch 49
epoch 50
epoch 51
epoch 52
epoch 53
epoch 54
epoch 55
epoch 56
epoch 57
epoch 58
epoch 59
epoch 60
epoch 61
epoch 62
epoch 63
epoch 64
epoch 65
epoch 66
epoch 67
epoch 68
epoch 69
epoch 70
epoch 71
epoch 72
epoch 73
epoch 74
epoch 75
epoch 76
epoch 77
epoch 78
epoch 79
epoch 80
epoch 81
epoch 82
epoch 83
epoch 84
epoch 85
epoch 86
epoch 87
epoch 88
epoch 89
epoch 90
epoch 91
epoch 92
epoch 93
epoch 94
epoch 95
epoch 96
epoch 97
epoch 98
epoch 99


In [8]:
# get test acc (using the trainning set partition not used to train earlier)
test_prob = network(x_train[train_set_size:])
test_class = torch.argmax(test_prob,dim=-1)
torch.sum(test_class == y_train[train_set_size:])/len(y_train[train_set_size:])

tensor(0.9795, device='cuda:0')

In [9]:
x_train = x_train[train_set_size:]
y_train = y_train[train_set_size:]

In [10]:
x_train_flat = [i.flatten().cpu().numpy() for i in x_train]

In [11]:
cluster_labels  = kmeans.fit_predict(x_train_flat)

/home/dezhang/anaconda3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [12]:
cluster_count = dict()
for i in cluster_labels:
    cluster_count[i] = cluster_count.get(i,0) + 1

In [13]:
# poison 5 smallest subpopulations
poison_cluster = sorted([i for i in cluster_count.items()], key=lambda x:x[1])[-5:]
n_poison_each_cluster = dict([(i[0], int(i[1]*alpha)) for i in poison_cluster])

In [14]:
n_poison_each_cluster

{8: 972, 93: 980, 4: 1064, 32: 1178, 58: 1238}

In [15]:
poison_cluster

[(8, 486), (93, 490), (4, 532), (32, 589), (58, 619)]

In [16]:
import random
random.seed(0)

new_x = []
new_y = []
new_y_correct=[]
for i in range(len(y_train)):
    cluster_y = cluster_labels[i]
    if cluster_y in n_poison_each_cluster and n_poison_each_cluster[cluster_y] > 0:
        while n_poison_each_cluster[cluster_y] > 0:
            new_y.append((y_train[i]+1)%10)
            new_x.append(x_train[i])
            new_y_correct.append(y_train[i])
            n_poison_each_cluster[cluster_y] = n_poison_each_cluster[cluster_y] - 1


In [17]:
x_train_poisoned = torch.concat((x_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_x])).to(device)))
y_train_poisoned = torch.concat((y_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_y])).to(device)))
y_train_correct =  torch.concat((y_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_y_correct])).to(device)))
# y_train_poisoned = y_train + new_y

In [18]:
x_train_poisoned = x_train_poisoned.to(device)
y_train_poisoned = y_train_poisoned.type(torch.long).to(device)

In [19]:
train_idx = [i for i in range(len(x_train_poisoned))]
import random
random.seed(0)
train_idx_shuffle = random.shuffle(train_idx)
x_train_poisoned = x_train_poisoned[train_idx_shuffle][0]
y_train_poisoned = y_train_poisoned[train_idx_shuffle][0]

In [20]:
# determine which of the test data falls into the cluster
x_test = [i[0] for i in test_loader]
y_test = [i[1] for i in test_loader]

In [21]:
x_poisoned_flat = [i.flatten().cpu().numpy() for i in x_train_poisoned]
poisoned_cluster_labels = kmeans.predict(x_poisoned_flat)

In [22]:
x_test_cluster = kmeans.predict([i.detach().cpu().numpy().flatten() for i in x_test])
x_test_in_poison = [int(i in n_poison_each_cluster) for i in x_test_cluster]

In [23]:
import statistics 
import copy
torch.manual_seed(1) # for reproducibility... 
n_epoch = 30
batch_size = 50
for _ in range(n_epoch):
    print("epoch " + str(_))
    
    for i in range(0,len(x_train_poisoned),batch_size):

        feature = x_train_poisoned[i:i+batch_size]
        target = y_train_poisoned[i:i+batch_size]
        optimizer.zero_grad()
        output = network(feature)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    
    
    
    with torch.no_grad():
        ncorrect_clean = 0
        ncorrect_poison = 0
        
        n_clean = 0
        n_poison = 0
        out_prob = network(torch.stack([i[0] for i in x_test]).to(device))
        out_class = torch.argmax(out_prob, dim=-1)
        for i in range(len(out_class)):
            pred_class = int(out_class[i])
            actual_class = int(y_test[i])
        
        
            if x_test_in_poison[i]:
                ncorrect_poison += int(pred_class == actual_class) 
                n_poison += 1
            else:
                ncorrect_clean += int(pred_class == actual_class)
                n_clean += 1
        clean_acc = ncorrect_clean/n_clean
        asr = 1-ncorrect_poison/n_poison # propotion we managed to misclassify as adversary class (either by chance or attack)
        print("ACC: ", clean_acc, "ASR", asr)

epoch 0
ACC:  0.11642778390297685 ASR 0.9946236559139785
epoch 1
ACC:  0.4836824696802646 ASR 0.9913978494623656
epoch 2
ACC:  0.6601984564498347 ASR 0.9870967741935484
epoch 3
ACC:  0.7603087100330761 ASR 0.975268817204301
epoch 4
ACC:  0.7778390297684675 ASR 0.9311827956989247
epoch 5
ACC:  0.7938257993384785 ASR 0.9440860215053763
epoch 6
ACC:  0.8424476295479603 ASR 0.7913978494623656
epoch 7
ACC:  0.8529217199558986 ASR 0.7827956989247312
epoch 8
ACC:  0.8124586549062844 ASR 0.878494623655914
epoch 9
ACC:  0.8859977949283352 ASR 0.6591397849462366
epoch 10
ACC:  0.8553472987872106 ASR 0.7548387096774194
epoch 11
ACC:  0.9089305402425579 ASR 0.5881720430107527
epoch 12
ACC:  0.8718853362734289 ASR 0.6924731182795698
epoch 13
ACC:  0.9093715545755237 ASR 0.5440860215053763
epoch 14
ACC:  0.9066152149944873 ASR 0.575268817204301
epoch 15
ACC:  0.9109151047409041 ASR 0.575268817204301
epoch 16
ACC:  0.9124586549062844 ASR 0.5494623655913978
epoch 17
ACC:  0.9238147739801543 ASR 0.4806